# 🤖 Lokaler Chatbot mit HuggingFace Transformers

Dieses Notebook lädt ein Sprachmodell direkt auf dein Gerät und stellt es als interaktiven Chatbot zur Verfügung.

**Empfohlene Modelle (nach verfügbarem RAM/VRAM):**

| Modell | Größe | RAM/VRAM | Qualität |
|---|---|---|---|
| `Qwen/Qwen3-0.6B` | ~1 GB | ≥ 4 GB RAM | Einstieg |
| `Qwen/Qwen3-1.7B` | ~3 GB | ≥ 6 GB RAM | Gut |
| `Qwen/Qwen3-4B` | ~8 GB | ≥ 10 GB RAM | Sehr gut |
| `Qwen/Qwen3-8B` | ~16 GB | ≥ 16 GB RAM | Exzellent |
| `meta-llama/Llama-3.2-3B` | ~6 GB | ≥ 8 GB RAM | Gut (Llama) |

> **Hinweis:** Für 4-Bit-Quantisierung (halber Speicherbedarf) `load_in_4bit=True` setzen (erfordert `bitsandbytes`).

## 1. Installation der benötigten Pakete

In [ ]:
# Nur einmalig ausführen
%pip install transformers accelerate torch ipywidgets

# Optional: für 4-Bit-Quantisierung (spart ca. 50% VRAM)
# !pip install bitsandbytes

## 2. Konfiguration

In [ ]:
# ============================================================
# HIER KONFIGURIEREN
# ============================================================

# Modell auswählen (siehe Tabelle oben)
MODEL_NAME = "Qwen/Qwen3-0.6B"

# Systemprompt: Persönlichkeit und Verhalten des Chatbots
SYSTEM_PROMPT = """Du bist ein hilfreicher Assistent. 
Antworte präzise und klar auf Deutsch. 
Bei Fachfragen erklärst du Sachverhalte verständlich und strukturiert."""

# 4-Bit-Quantisierung aktivieren? (benötigt bitsandbytes)
USE_4BIT = False

# Maximale Länge der Antwort (in Tokens)
MAX_NEW_TOKENS = 512

# Temperatur: 0.0 = deterministisch, 1.0 = kreativ
TEMPERATURE = 0.7

# ============================================================

## 3. Modell laden

⏳ Dieser Schritt kann beim ersten Ausführen einige Minuten dauern (Download des Modells).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

print(f"📦 Lade Modell: {MODEL_NAME}")
print(f"🖥️  CUDA verfügbar: {torch.cuda.is_available()}")

# Gerät bestimmen
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️  Gerät: {device}")

# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quantisierungskonfiguration (optional)
quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

# Modell laden
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    quantization_config=quantization_config,
)

if device == "cpu":
    model = model.to(device)

model.eval()
print(f"\n✅ Modell erfolgreich geladen!")
print(f"📊 Parameter: {sum(p.numel() for p in model.parameters()) / 1e9:.1f} Mrd.")

## 4. Chatbot-Logik

In [ ]:
def generate_response(messages: list, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """
    Erzeugt eine Antwort des Modells auf Basis des Gesprächsverlaufs.
    
    Args:
        messages: Liste von Nachrichten im Format [{"role": ..., "content": ...}]
        max_new_tokens: Maximale Anzahl neuer Tokens
    
    Returns:
        Antwort als String
    """
    # Chat-Template anwenden (falls vorhanden)
    try:
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        # Fallback: manuelles Format
        input_text = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            if role == "system":
                input_text += f"<|system|>\n{content}\n"
            elif role == "user":
                input_text += f"<|user|>\n{content}\n"
            elif role == "assistant":
                input_text += f"<|assistant|>\n{content}\n"
        input_text += "<|assistant|>\n"

    # Tokenisieren
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]

    # Generieren
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE,
            do_sample=TEMPERATURE > 0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Nur neue Tokens dekodieren (ohne Input)
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


# Gesprächsverlauf (global)
chat_history = []

def reset_chat():
    """Setzt den Gesprächsverlauf zurück."""
    global chat_history
    chat_history = []

print("✅ Chatbot-Logik bereit.")

## 5. Interaktive Chat-Oberfläche

▶️ Zelle ausführen → Eingabefeld erscheint unten

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Styles ────────────────────────────────────────────────────────────────
CHAT_CSS = """
<style>
.chat-container {
    font-family: 'Segoe UI', system-ui, sans-serif;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
}
.chat-header {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    color: #e0e0e0;
    padding: 16px 20px;
    border-radius: 12px 12px 0 0;
    font-size: 15px;
    font-weight: 600;
    letter-spacing: 0.5px;
    border-bottom: 2px solid #e94560;
    display: flex;
    align-items: center;
    gap: 10px;
}
.chat-messages {
    background: #f8f9fc;
    border: 1px solid #e0e4ef;
    border-top: none;
    min-height: 350px;
    max-height: 450px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 12px;
}
.msg-row {
    display: flex;
    gap: 10px;
    align-items: flex-start;
    animation: fadeIn 0.3s ease;
}
@keyframes fadeIn { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: translateY(0); } }
.msg-row.user { flex-direction: row-reverse; }
.avatar {
    width: 34px; height: 34px;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 16px; flex-shrink: 0;
}
.avatar.bot { background: linear-gradient(135deg, #0f3460, #e94560); color: white; }
.avatar.user { background: linear-gradient(135deg, #2d6a4f, #52b788); color: white; }
.bubble {
    max-width: 82%;
    padding: 10px 14px;
    border-radius: 16px;
    font-size: 14px;
    line-height: 1.55;
    white-space: pre-wrap;
    word-break: break-word;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08);
}
.bubble.bot {
    background: white;
    color: #1a1a2e;
    border: 1px solid #e0e4ef;
    border-bottom-left-radius: 4px;
}
.bubble.user {
    background: linear-gradient(135deg, #0f3460, #1a4a8a);
    color: #f0f4ff;
    border-bottom-right-radius: 4px;
}
.bubble.thinking {
    background: #fff8e1;
    color: #7a6000;
    border: 1px dashed #f0c040;
    font-style: italic;
    font-size: 13px;
}
.timestamp {
    font-size: 10px;
    color: #aaa;
    margin-top: 3px;
    text-align: right;
}
.msg-row.user .timestamp { text-align: left; }
.empty-state {
    text-align: center;
    color: #b0b8cc;
    padding: 40px 20px;
    font-size: 13px;
    flex: 1;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 8px;
}
.empty-icon { font-size: 36px; }
.info-bar {
    background: #eef0f8;
    border: 1px solid #e0e4ef;
    border-top: none;
    border-bottom: none;
    padding: 6px 16px;
    font-size: 11px;
    color: #7a82a0;
    display: flex;
    justify-content: space-between;
}
</style>
"""

# ── Widgets ───────────────────────────────────────────────────────────────
output_area   = widgets.Output()
input_box     = widgets.Text(
    placeholder="Nachricht eingeben …",
    layout=widgets.Layout(width="84%", height="38px")
)
send_btn      = widgets.Button(
    description="Senden",
    button_style="primary",
    icon="paper-plane",
    layout=widgets.Layout(width="14%", height="38px")
)
reset_btn     = widgets.Button(
    description="Neu",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="10%", height="32px")
)
status_label  = widgets.Label(value="Bereit")
input_row     = widgets.HBox(
    [input_box, send_btn],
    layout=widgets.Layout(
        width="100%",
        padding="10px 16px",
        background="white",
        border="1px solid #e0e4ef",
        border_top="none",
        border_radius="0 0 12px 12px",
        gap="8px"
    )
)

# ── Hilfsfunktionen ───────────────────────────────────────────────────────
import datetime

def now_str():
    return datetime.datetime.now().strftime("%H:%M")

def render_messages():
    """Rendert den kompletten Chatverlauf als HTML."""
    if not chat_history:
        return """
        <div class="empty-state">
            <span class="empty-icon">🤖</span>
            <strong>Chatbot bereit</strong>
            <span>Stelle eine Frage – ich antworte direkt auf deinem Gerät.</span>
        </div>
        """
    html = ""
    for msg in chat_history:
        role = msg["role"]
        if role == "system":
            continue
        content = msg["content"].replace("<", "&lt;").replace(">", "&gt;")
        ts      = msg.get("ts", "")
        if role == "user":
            html += f"""
            <div class="msg-row user">
                <div>
                    <div class="bubble user">{content}</div>
                    <div class="timestamp">{ts}</div>
                </div>
                <div class="avatar user">👤</div>
            </div>"""
        else:
            html += f"""
            <div class="msg-row bot">
                <div class="avatar bot">🤖</div>
                <div>
                    <div class="bubble bot">{content}</div>
                    <div class="timestamp">{ts}</div>
                </div>
            </div>"""
    return html

def refresh_display(thinking=False):
    """Aktualisiert die Ausgabe."""
    msgs = render_messages()
    thinking_html = ""
    if thinking:
        thinking_html = """
        <div class="msg-row bot">
            <div class="avatar bot">🤖</div>
            <div class="bubble thinking">⏳ Denkt nach …</div>
        </div>"""

    scroll_js = """
    <script>
        var el = document.getElementById('chat-msgs');
        if (el) el.scrollTop = el.scrollHeight;
    </script>"""

    full_html = f"""
    {CHAT_CSS}
    <div class="chat-container">
        <div class="chat-header">
            <span>🤖</span>
            <span>Lokaler Chatbot &nbsp;·&nbsp; <small style='font-weight:400;opacity:.8'>{MODEL_NAME.split('/')[-1]}</small></span>
        </div>
        <div class="chat-messages" id="chat-msgs">
            {msgs}
            {thinking_html}
        </div>
        <div class="info-bar">
            <span>💬 {len([m for m in chat_history if m['role'] != 'system'])} Nachrichten</span>
            <span>Temp: {TEMPERATURE} &nbsp;|&nbsp; Max Tokens: {MAX_NEW_TOKENS}</span>
        </div>
    </div>
    {scroll_js}
    """
    with output_area:
        clear_output(wait=True)
        display(HTML(full_html))

# ── Event-Handler ─────────────────────────────────────────────────────────
def on_send(event=None):
    user_text = input_box.value.strip()
    if not user_text:
        return

    input_box.value = ""
    send_btn.disabled = True
    input_box.disabled = True

    # System-Prompt beim ersten Mal einfügen
    if not chat_history:
        chat_history.append({"role": "system", "content": SYSTEM_PROMPT})

    # Nutzernachricht
    chat_history.append({"role": "user", "content": user_text, "ts": now_str()})
    refresh_display(thinking=True)

    # Modell aufrufen (nur role+content übergeben)
    api_messages = [{"role": m["role"], "content": m["content"]} for m in chat_history]
    try:
        response = generate_response(api_messages)
    except Exception as e:
        response = f"❌ Fehler: {e}"

    chat_history.append({"role": "assistant", "content": response, "ts": now_str()})
    refresh_display(thinking=False)

    send_btn.disabled = False
    input_box.disabled = False
    input_box.focus()

def on_reset(event=None):
    reset_chat()
    refresh_display()

# Enter-Taste senden
input_box.on_submit(on_send)
send_btn.on_click(on_send)
reset_btn.on_click(on_reset)

# ── Layout zusammenbauen & anzeigen ───────────────────────────────────────
controls_row = widgets.HBox(
    [reset_btn, status_label],
    layout=widgets.Layout(justify_content="flex-start", gap="8px", margin="6px 0 0 0")
)

display(output_area, input_row, controls_row)
refresh_display()

---
## 6. Optionaler Test ohne GUI

In [ ]:
# Direkte Anfrage – nützlich zum Testen
test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Erkläre in zwei Sätzen, was ein Transformer-Modell ist."}
]

antwort = generate_response(test_messages)
print("Antwort:", antwort)

---
## Hinweise zur Verwendung im Unterricht

- **Datenschutz:** Alle Daten bleiben auf dem eigenen Gerät – kein API-Key, keine Cloud.
- **Systemprompt anpassen:** In Zelle 2 (`SYSTEM_PROMPT`) kann die Rolle des Chatbots frei definiert werden, z.B. als Fachtutor für ein bestimmtes Thema.
- **Modell wechseln:** `MODEL_NAME` in Zelle 2 ändern, dann ab Zelle 3 neu ausführen.
- **Gesprächsverlauf:** Wird im Arbeitsspeicher gehalten. `reset_chat()` oder den **Neu**-Button verwenden, um neu zu starten.
- **Erweiterungsideen:**
  - Gesprächsprotokoll als `.txt` exportieren
  - RAG-Pipeline mit Schulbuch-PDFs ergänzen
  - Streaming-Ausgabe mit `TextIteratorStreamer`